# Architecture

## Tokenisation + Data Loading

In [ ]:
import os

from socrates_ai.data_prep import TheFarmer
from socrates_ai.model_prep import Translator

farmer = TheFarmer()
tokeniser_path = os.path.join('..', 'data', 'tokenizer.json')

translator = Translator(cleaned_books_path = farmer.clean_dir)
oracle_tokens = translator.load_tokeniser(path=tokeniser_path)

In [ ]:
import torch

from socrates_ai.helpers.training_helpers import load_book_paths, split_books_train_val

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

book_paths = load_book_paths(farmer.clean_dir)
print(f"{len(book_paths)} cleaned books found")

train_ids, val_ids = split_books_train_val(book_paths, oracle_tokens)
print(f"train tokens: {len(train_ids):,}")
print(f"val tokens:   {len(val_ids):,}")

Using device: cpu
634 cleaned books found
train tokens: 56,208,830
val tokens:   7,460,394


# Model

In [ ]:
from socrates_ai.model_prep import MiniTransformer

model = MiniTransformer(
    vocab_size=oracle_tokens.get_vocab_size(),
    d_model=256,
    n_layers=4,
    n_heads=4,
    block_size=128,
    dropout=0.1,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params:,}")


Total parameters: 7,288,320


## Training

Three pieces: a helper to measure loss without training on it (`estimate_loss`), the actual training loop (`train_model`), and a plot of how loss moved over the run.

# Training Model Class

In [4]:
from socrates_ai.training import TrainingSocrates

In [ ]:
# reuses the `model` instantiated in the sanity-check cell above -- still
# untrained at this point unless you call load_checkpoint() below
trainer = TrainingSocrates(model, train_ids, val_ids, device=device)


# to actually train (see the runtime note earlier in the notebook for how
# long this takes on this hardware):
trainer.train(max_steps=400, eval_interval=200, eval_iters=15)
# trainer.plot_training_curves()
# print(trainer.talk(oracle_tokens, "free will is", max_new_tokens=60, temperature=0.8))

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


step      1 | lr 1.50e-06 | train loss 9.6961 | val loss 9.6989
step    200 | lr 3.00e-04 | train loss 6.2411 | val loss 6.3063


In [ ]:
# to resume a previous session instead of starting fresh, uncomment:
# trainer.load_checkpoint()  # defaults to the most recent checkpoint

## Generation

To answer the question directly first: **yes, we need an explicit cap on output length.** Generation is autoregressive — feed in a prompt, predict the next token, append it, feed the whole thing back in, predict again — and nothing about that loop naturally terminates on its own. Without a hard stop, it would just keep generating forever. `max_new_tokens` below is that hard stop.

There's also a second, more subtle limit specific to this architecture: `pos_emb` only has `block_size=128` rows (from a few turns back), so the model *cannot* be handed more than 128 tokens of context at once — there's no positional embedding for position 128+. As the generated sequence grows past that, `generate` below always crops back to the most recent `block_size` tokens before each forward pass ("sliding window"). Your old GRU's `predict()` didn't need this — it only ever fed in one token at a time plus a carried hidden state `h`, so context length was never bounded. A transformer has no hidden state to carry, so every step re-processes actual token positions, and that's exactly what makes the `block_size` cap necessary.

As a softer, optional stop: we trained `<eos>` to mark the end of a book, so if the model generates that token, `generate` can stop early rather than padding out to `max_new_tokens` regardless. Early in training this signal won't be reliable yet, but it costs nothing to have in place.

In [15]:
@torch.no_grad()  # generation is inference only -- no gradients needed
def generate(model, idx, max_new_tokens, block_size, device, temperature=1.0, eos_id=None):
    """
    Autoregressively extend `idx` (a (1, T) tensor of token ids) by up to
    max_new_tokens new tokens. Stops early if `eos_id` is produced.
    """
    model.eval()  # turn dropout off for generation, same reason as estimate_loss

    for _ in range(max_new_tokens):
        # crop to the last block_size tokens -- the model has no positional
        # embedding for anything beyond that, so this is required, not optional
        idx_cond = idx if idx.size(1) <= block_size else idx[:, -block_size:]

        logits, _ = model(idx_cond)  # (1, T, vocab_size)

        # only the LAST position's logits matter -- that's the prediction for
        # "what comes after everything we've fed in so far"
        last_logits = logits[:, -1, :] / temperature
        # temperature < 1 sharpens the distribution (more confident/repetitive),
        # > 1 flattens it (more random/varied) -- dividing before softmax is
        # what actually changes the shape of the resulting probabilities

        probs = F.softmax(last_logits, dim=-1)

        # sample rather than always taking the top prediction (argmax) -- greedy
        # decoding tends to get stuck in repetitive loops; sampling from the
        # full distribution gives varied, more natural-looking output
        next_id = torch.multinomial(probs, num_samples=1)

        idx = torch.cat([idx, next_id], dim=1)  # append and feed back in next iteration

        if eos_id is not None and next_id.item() == eos_id:
            break  # model signalled "this piece of writing is done"

    model.train()  # hand control back in training mode, matching estimate_loss's convention
    return idx

In [62]:
def talk(model, tokeniser, prompt, max_new_tokens=100, block_size=128, device="cpu", temperature=1.0, stop_at_eos=True):
    """
    User-facing wrapper: raw prompt string in, generated continuation string
    out. Handles the encode -> generate -> decode round trip so `generate`
    itself only has to deal with token ids.
    """
    prompt_ids = tokeniser.encode(prompt).ids
    idx = torch.tensor([prompt_ids], dtype=torch.long, device=device)  # (1, T) -- batch of 1

    eos_id = tokeniser.token_to_id("<eos>") if stop_at_eos else None

    out_ids = generate(model, idx, max_new_tokens, block_size, device, temperature, eos_id)
    return tokeniser.decode(out_ids[0].tolist())


print(talk(model, oracle_tokens, "free will is", max_new_tokens=60, block_size=128, device=device, temperature=0.8))

 free will is
for our souls who have all self-offards our children of
the power-willings with mankind, we may be created with
true and to me and so’s
and he would look from the great people. So not too, in this
Gmpors, in the


# Benchmarks

Model trained on 50 books, 8000 token vocan size, outputs did not make sense, no real structure, some words were formed correctly

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
step     1 | train loss 8.7233 | val loss 8.7556
step    30 | train loss 6.9954 | val loss 7.1738
step    60 | train loss 6.5458 | val loss 6.8526
step    90 | train loss 6.3036 | val loss 6.6178
step   120 | train loss 6.0484 | val loss 6.3679
step   150 | train loss 5.8711 | val loss 6.1601
step   180 | train loss 5.7081 | val loss 6.0760
step   210 | train loss 5.6247 | val loss 5.9218
step   240 | train loss 5.5371 | val loss 5.8338
step   270 | train loss 5.4553 | val loss 5.7892
step   300 | train loss 5.4088 | val loss 5.7817

Training finished in 3693.6s